In [1]:
import pandas as pd

from cubes.construct.buildingconfig import load_building_config
from cubes.construct.building import Building
from cubes.package.envconfig import EnvConfig
from cubes.package.simple_simulation import prepare_simulation



files_dir = "/workspaces/CUBES/exp/jack/paper/misc/input"
bc_file_path = "/workspaces/CUBES/exp/jack/paper/thermostat_experiment/input/paper/2023/case0.json" #"/workspaces/CUBES/exp/jack/paper/thermostat_experiment/input/test_case_H28/zone0.json"


materials = pd.read_pickle("/workspaces/CUBES/cubes/materials.pickle")
windows = pd.read_pickle("/workspaces/CUBES/cubes/windows.pickle")

BC = load_building_config(bc_file_path, files_dir=files_dir)
EC = EnvConfig(files_dir=files_dir)
building=Building(BC,materials,windows)
#building=Building(BC,materials_evaluator(),windows_evaluator())
building.build()
idf = building.get_idf()

# idf.view_model()


<jemalloc>: MADV_DONTNEED does not work (memset will be used instead)
<jemalloc>: (This is the expected behaviour if you are running under QEMU)


In [2]:
idf.save("test.idf")

In [12]:
def add_back_room_gains_and_schedules(self):
    """Add schedules and internal gains for the back room (dining room)."""

    # Morning Schedule (08:00–08:30 on weekdays, 09:30–10:00 on weekends)
    self.idf.newidfobject(
        "SCHEDULE:COMPACT",
        Name="BackRoom-MorningGains-Schedule",
        Schedule_Type_Limits_Name="Any Number",
        Field_1="Through: 12/31",
        Field_2="For: Weekdays",
        Field_3="Until: 08:00, 0",
        Field_4="Until: 08:30, 460",  # Total actual gains (Hot food)
        Field_5="Until: 24:00, 0",
        Field_6="For: Weekends",
        Field_7="Until: 09:30, 0",
        Field_8="Until: 10:00, 460",  # Total actual gains (Hot food)
        Field_9="Until: 24:00, 0",
    )

    # Evening Schedule (17:00–18:00)
    self.idf.newidfobject(
        "SCHEDULE:COMPACT",
        Name="BackRoom-EveningGains-Schedule",
        Schedule_Type_Limits_Name="Any Number",
        Field_1="Through: 12/31",
        Field_2="For: Weekdays Weekends",
        Field_3="Until: 17:00, 0",
        Field_4="Until: 18:00, 480",  # Total actual gains (Hot food + Lighting)
        Field_5="Until: 24:00, 0",
    )

    # Add Morning Gains (Hot Food)
    self.idf.newidfobject(
        "ELECTRICEQUIPMENT",
        Name="BackRoom-HotFood-Morning",
        Zone_or_ZoneList_Name="backroom",
        Schedule_Name="BackRoom-MorningGains-Schedule",
        Design_Level_Calculation_Method="LightingLevel",
        Design_Level=460,  # Total actual gains
        Fraction_Latent=0.0,
        Fraction_Radiant=0.3,  # Adjust as needed
        Fraction_Lost=0.1,  # Adjust as needed
        EndUse_Subcategory="Cooking",
    )

    # Add Evening Gains (Hot Food)
    self.idf.newidfobject(
        "ELECTRICEQUIPMENT",
        Name="BackRoom-HotFood-Evening",
        Zone_or_ZoneList_Name="backroom",
        Schedule_Name="BackRoom-EveningGains-Schedule",
        Design_Level_Calculation_Method="LightingLevel",
        Design_Level=450,  # Hot food portion of the gains
        Fraction_Latent=0.0,
        Fraction_Radiant=0.3,  # Adjust as needed
        Fraction_Lost=0.1,  # Adjust as needed
        EndUse_Subcategory="Cooking",
    )

    # Add Evening Gains (Lighting)
    self.idf.newidfobject(
        "LIGHTS",
        Name="BackRoom-Lighting-Evening",
        Zone_or_ZoneList_Name="backroom",
        Schedule_Name="BackRoom-EveningGains-Schedule",
        Design_Level_Calculation_Method="LightingLevel",
        Lighting_Level=30,  # Lighting power in watts
        Fraction_Radiant=0.7,
        Fraction_Visible=0.3,
        Fraction_Replaceable=1.0,
        EndUse_Subcategory="Lighting",
    )

def add_bedroom1_gains_and_schedules(self):
    """Add schedules and internal gains for Bedroom 1."""

    # Afternoon Schedule (16:00–17:00)
    self.idf.newidfobject(
        "SCHEDULE:COMPACT",
        Name="Bedroom1-AfternoonGains-Schedule",
        Schedule_Type_Limits_Name="Any Number",
        Field_1="Through: 12/31",
        Field_2="For: Weekdays Weekends",
        Field_3="Until: 16:00, 0",
        Field_4="Until: 17:00, 200",  # Total actual gains (Lighting + others)
        Field_5="Until: 24:00, 0",
    )

    # Evening Schedule (19:00–20:00)
    self.idf.newidfobject(
        "SCHEDULE:COMPACT",
        Name="Bedroom1-EveningGains-Schedule",
        Schedule_Type_Limits_Name="Any Number",
        Field_1="Through: 12/31",
        Field_2="For: Weekdays Weekends",
        Field_3="Until: 19:00, 0",
        Field_4="Until: 20:00, 120",  # Total actual gains (Lighting + others)
        Field_5="Until: 24:00, 0",
    )

    # Late Evening Schedule (20:00–22:30)
    self.idf.newidfobject(
        "SCHEDULE:COMPACT",
        Name="Bedroom1-LateEveningGains-Schedule",
        Schedule_Type_Limits_Name="Any Number",
        Field_1="Through: 12/31",
        Field_2="For: Weekdays Weekends",
        Field_3="Until: 20:00, 0",
        Field_4="Until: 22:30, 300",  # Total actual gains (Lighting + Computer)
        Field_5="Until: 24:00, 0",
    )

    # Add Gains for Afternoon
    self.idf.newidfobject(
        "LIGHTS",
        Name="Bedroom1-Lighting-Afternoon",
        Zone_or_ZoneList_Name="bedroom_1",
        Schedule_Name="Bedroom1-AfternoonGains-Schedule",
        Design_Level_Calculation_Method="LightingLevel",
        Lighting_Level=30,  # Lighting power in watts
        Fraction_Radiant=0.7,
        Fraction_Visible=0.3,
        Fraction_Replaceable=1.0,
        EndUse_Subcategory="Lighting",
    )

    # Add Gains for Evening
    self.idf.newidfobject(
        "LIGHTS",
        Name="Bedroom1-Lighting-Evening",
        Zone_or_ZoneList_Name="bedroom_1",
        Schedule_Name="Bedroom1-EveningGains-Schedule",
        Design_Level_Calculation_Method="LightingLevel",
        Lighting_Level=30,  # Lighting power in watts
        Fraction_Radiant=0.7,
        Fraction_Visible=0.3,
        Fraction_Replaceable=1.0,
        EndUse_Subcategory="Lighting",
    )

    # Add Gains for Late Evening (Lighting)
    self.idf.newidfobject(
        "LIGHTS",
        Name="Bedroom1-Lighting-LateEvening",
        Zone_or_ZoneList_Name="bedroom_1",
        Schedule_Name="Bedroom1-LateEveningGains-Schedule",
        Design_Level_Calculation_Method="LightingLevel",
        Lighting_Level=30,  # Lighting power in watts
        Fraction_Radiant=0.7,
        Fraction_Visible=0.3,
        Fraction_Replaceable=1.0,
        EndUse_Subcategory="Lighting",
    )

    # Add Gains for Late Evening (Computer)
    self.idf.newidfobject(
        "ELECTRICEQUIPMENT",
        Name="Bedroom1-Computer-LateEvening",
        Zone_or_ZoneList_Name="bedroom_1",
        Schedule_Name="Bedroom1-LateEveningGains-Schedule",
        Design_Level_Calculation_Method="LightingLevel",
        Design_Level=100,  # Computer power in watts
        Fraction_Latent=0.0,
        Fraction_Radiant=0.4,  # Adjust as needed
        Fraction_Lost=0.2,  # Adjust as needed
        EndUse_Subcategory="Computer",
    )


def add_front_room_gains_and_schedules(self):
    """Add schedules and internal gains for the front room (living room)."""

    # Early Evening Schedule (18:00–19:00)
    self.idf.newidfobject(
        "SCHEDULE:COMPACT",
        Name="FrontRoom-EarlyEveningGains-Schedule",
        Schedule_Type_Limits_Name="Any Number",
        Field_1="Through: 12/31",
        Field_2="For: Weekdays Weekends",
        Field_3="Until: 18:00, 0",
        Field_4="Until: 19:00, 580",  # Total actual gains (TV: 150 W + Lighting: 30 W + others)
        Field_5="Until: 24:00, 0",
    )

    # Late Evening Schedule (19:00–22:30)
    self.idf.newidfobject(
        "SCHEDULE:COMPACT",
        Name="FrontRoom-LateEveningGains-Schedule",
        Schedule_Type_Limits_Name="Any Number",
        Field_1="Through: 12/31",
        Field_2="For: Weekdays Weekends",
        Field_3="Until: 19:00, 0",
        Field_4="Until: 22:30, 400",  # Total actual gains (TV: 150 W + Lighting: 30 W + others)
        Field_5="Until: 24:00, 0",
    )

    # Add TV Gains for Early Evening
    self.idf.newidfobject(
        "ELECTRICEQUIPMENT",
        Name="FrontRoom-TV-EarlyEvening",
        Zone_or_ZoneList_Name="front_room",
        Schedule_Name="FrontRoom-EarlyEveningGains-Schedule",
        Design_Level_Calculation_Method="LightingLevel",
        Design_Level=150,  # TV power in watts
        Fraction_Latent=0.0,
        Fraction_Radiant=0.2,  # Adjust as needed
        Fraction_Lost=0.1,  # Adjust as needed
        EndUse_Subcategory="TV",
    )

    # Add TV Gains for Late Evening
    self.idf.newidfobject(
        "ELECTRICEQUIPMENT",
        Name="FrontRoom-TV-LateEvening",
        Zone_or_ZoneList_Name="front_room",
        Schedule_Name="FrontRoom-LateEveningGains-Schedule",
        Design_Level_Calculation_Method="LightingLevel",
        Design_Level=150,  # TV power in watts
        Fraction_Latent=0.0,
        Fraction_Radiant=0.2,  # Adjust as needed
        Fraction_Lost=0.1,  # Adjust as needed
        EndUse_Subcategory="TV",
    )

    # Add Lighting Gains for Early Evening
    self.idf.newidfobject(
        "LIGHTS",
        Name="FrontRoom-Lighting-EarlyEvening",
        Zone_or_ZoneList_Name="front_room",
        Schedule_Name="FrontRoom-EarlyEveningGains-Schedule",
        Design_Level_Calculation_Method="LightingLevel",
        Lighting_Level=30,  # Lighting power in watts
        Fraction_Radiant=0.7,
        Fraction_Visible=0.3,
        Fraction_Replaceable=1.0,
        EndUse_Subcategory="Lighting",
    )

    # Add Lighting Gains for Late Evening
    self.idf.newidfobject(
        "LIGHTS",
        Name="FrontRoom-Lighting-LateEvening",
        Zone_or_ZoneList_Name="front_room",
        Schedule_Name="FrontRoom-LateEveningGains-Schedule",
        Design_Level_Calculation_Method="LightingLevel",
        Lighting_Level=30,  # Lighting power in watts
        Fraction_Radiant=0.7,
        Fraction_Visible=0.3,
        Fraction_Replaceable=1.0,
        EndUse_Subcategory="Lighting",
    )

def add_kitchen_gains_and_schedules(self):
    """Add schedules and internal gains for the kitchen."""

    # Morning Cooking Schedule
    self.idf.newidfobject(
        "SCHEDULE:COMPACT",
        Name="Kitchen-MorningCooking-Schedule",
        Schedule_Type_Limits_Name="Any Number",
        Field_1="Through: 12/31",
        Field_2="For: Weekdays",
        Field_3="Until: 07:30, 0",
        Field_4="Until: 08:00, 160",  # Morning cooking
        Field_5="Until: 24:00, 0",
        Field_6="For: Weekends",
        Field_7="Until: 09:00, 0",
        Field_8="Until: 09:30, 160",  # Morning cooking
        Field_9="Until: 24:00, 0",
    )

    # Evening Cooking Schedule
    self.idf.newidfobject(
        "SCHEDULE:COMPACT",
        Name="Kitchen-EveningCooking-Schedule",
        Schedule_Type_Limits_Name="Any Number",
        Field_1="Through: 12/31",
        Field_2="For: Weekdays Weekends",
        Field_3="Until: 16:00, 0",
        Field_4="Until: 17:00, 1600",  # Evening cooking
        Field_5="Until: 24:00, 0",
    )

    # Lighting Schedule
    self.idf.newidfobject(
        "SCHEDULE:COMPACT",
        Name="Kitchen-Lighting-Schedule",
        Schedule_Type_Limits_Name="Any Number",
        Field_1="Through: 12/31",
        Field_2="For: Weekdays Weekends",
        Field_3="Until: 16:00, 0",
        Field_4="Until: 17:00, 54",  # Lighting
        Field_5="Until: 24:00, 0",
    )

    # Fridge Schedule
    self.idf.newidfobject(
        "SCHEDULE:COMPACT",
        Name="Kitchen-Fridge-Schedule",
        Schedule_Type_Limits_Name="Any Number",
        Field_1="Through: 12/31",
        Field_2="For: AllDays",
        Field_3="Until: 24:00, 60",  # Fridge (constant all day)
    )

    # Add Morning Cooking Gains
    self.idf.newidfobject(
        "ELECTRICEQUIPMENT",
        Name="Kitchen-MorningCooking",
        Zone_or_ZoneList_Name="kitchen",
        Schedule_Name="Kitchen-MorningCooking-Schedule",
        Design_Level_Calculation_Method="LightingLevel",
        Design_Level=160,
        Fraction_Latent=0.0,
        Fraction_Radiant=0.3,  # Adjust as needed
        Fraction_Lost=0.1,  # Adjust as needed
        EndUse_Subcategory="Cooking",
    )

    # Add Evening Cooking Gains
    self.idf.newidfobject(
        "ELECTRICEQUIPMENT",
        Name="Kitchen-EveningCooking",
        Zone_or_ZoneList_Name="kitchen",
        Schedule_Name="Kitchen-EveningCooking-Schedule",
        Design_Level_Calculation_Method="LightingLevel",
        Design_Level=1600,
        Fraction_Latent=0.0,
        Fraction_Radiant=0.3,  # Adjust as needed
        Fraction_Lost=0.1,  # Adjust as needed
        EndUse_Subcategory="Cooking",
    )

    # Add Lighting Gains
    self.idf.newidfobject(
        "LIGHTS",
        Name="Kitchen-Lighting",
        Zone_or_ZoneList_Name="kitchen",
        Schedule_Name="Kitchen-Lighting-Schedule",
        Design_Level_Calculation_Method="LightingLevel",
        Lighting_Level=54,
        Fraction_Radiant=0.7,
        Fraction_Visible=0.3,
        Fraction_Replaceable=1.0,
        EndUse_Subcategory="Lighting",
    )

    # Add Fridge Gains
    self.idf.newidfobject(
        "ELECTRICEQUIPMENT",
        Name="Kitchen-Fridge",
        Zone_or_ZoneList_Name="kitchen",
        Schedule_Name="Kitchen-Fridge-Schedule",
        Design_Level_Calculation_Method="LightingLevel",
        Design_Level=60,
        Fraction_Latent=0.0,
        Fraction_Radiant=0.2,
        Fraction_Lost=0.2,
        EndUse_Subcategory="Fridge",
    )




In [9]:
add_kitchen_gains_and_schedules(idf)
add_front_room_gains_and_schedules(idf)
add_back_room_gains_and_schedules(idf)
add_bedroom1_gains_and_schedules(idf)


In [2]:
idf.save("test.idf")

In [ ]:
for zone in idf.idfobjects["ZONE"]:
    idf.newidfobject(
        "AirflowNetwork:MultiZone:Zone".upper(),
        Zone_Name=zone.Name,  # The name of the thermal zone
        Ventilation_Control_Mode="NoVent",  # Cracks operate passively; no active ventilation control
        Venting_Availability_Schedule_Name="Always",  # Schedule is mandatory but irrelevant for cracks
    )


In [2]:
idf.newidfobject(
        "AirflowNetwork:SimulationControl".upper(),
        Name="AFNControl",
        AirflowNetwork_Control="MultizoneWithoutDistribution",
        Wind_Pressure_Coefficient_Type="SurfaceAverageCalculation",
        Height_Selection_for_Local_Wind_Pressure_Calculation="OpeningHeight",
        Building_Type="LOWRISE",
        Maximum_Number_of_Iterations=500,
        Initialization_Type="ZeroNodePressures",
        Relative_Airflow_Convergence_Tolerance=1.0E-04,
        Absolute_Airflow_Convergence_Tolerance=1.0E-06,
        Convergence_Acceleration_Limit=-0.5,
        Azimuth_Angle_of_Long_Axis_of_Building=0.0,
        Ratio_of_Building_Width_Along_Short_Axis_to_Width_Along_Long_Axis=1.0,
    )



AIRFLOWNETWORK:SIMULATIONCONTROL,
    AFNControl,               !- Name
    MultizoneWithoutDistribution,    !- AirflowNetwork Control
    SurfaceAverageCalculation,    !- Wind Pressure Coefficient Type
    OpeningHeight,            !- Height Selection for Local Wind Pressure Calculation
    LOWRISE,                  !- Building Type
    500,                      !- Maximum Number of Iterations
    ZeroNodePressures,        !- Initialization Type
    0.0001,                   !- Relative Airflow Convergence Tolerance
    1e-06,                    !- Absolute Airflow Convergence Tolerance
    -0.5,                     !- Convergence Acceleration Limit
    0,                        !- Azimuth Angle of Long Axis of Building
    1,                        !- Ratio of Building Width Along Short Axis to Width Along Long Axis
    No,                       !- Height Dependence of External Node Temperature
    SkylineLU,                !- Solver
    No;                       !- Allow Unsupporte

In [4]:
# Define reusable crack templates with properties
crack_definitions = {
    "ExternalWallCrack": {"Cq": 0.00015, "n": 0.65},
    "InternalWallCrack": {"Cq": 0.00010, "n": 0.65},
    "FloorCeilingCrack": {"Cq": 0.00012, "n": 0.65},
    "RoofCrack": {"Cq": 0.00008, "n": 0.65},
}

# Add crack templates to the IDF
for crack_name, properties in crack_definitions.items():
    idf.newidfobject(
        "AirflowNetwork:MultiZone:Surface:Crack".upper(),
        Name=crack_name,
        Air_Mass_Flow_Coefficient_at_Reference_Conditions=properties["Cq"],
        Air_Mass_Flow_Exponent=properties["n"],
    )

# Helper function to classify surface types
def classify_surface(surface_type, boundary_condition):
    if surface_type.lower() == "wall":
        return "ExternalWallCrack" if boundary_condition.lower() == "outdoors" else "InternalWallCrack"
    elif surface_type.lower() in ["floor", "ceiling"]:
        return "FloorCeilingCrack"
    elif surface_type.lower() == "roof":
        return "RoofCrack"
    else:
        return None

# Keep track of processed surface pairs to avoid duplication
processed_surface_pairs = set()

# Loop through all BuildingSurface:Detailed objects
for surface in idf.idfobjects["BUILDINGSURFACE:DETAILED"]:
    surface_name = surface.Name
    zone_name = surface.Zone_Name
    surface_type = surface.Surface_Type
    boundary_condition = surface.Outside_Boundary_Condition
    boundary_object = surface.Outside_Boundary_Condition_Object

    # Avoid duplication for shared surfaces
    if boundary_condition.lower() == "surface" and boundary_object:
        # Create a unique key for the surface pair (order-independent)
        surface_pair = tuple(sorted([surface_name, boundary_object]))
        if surface_pair in processed_surface_pairs:
            continue  # Skip if this pair has already been processed
        processed_surface_pairs.add(surface_pair)

    # Classify the surface and get the appropriate crack template
    crack_name = classify_surface(surface_type, boundary_condition)
    if not crack_name:
        continue  # Skip surfaces that don't match any category

    # Add the AirflowNetwork:MultiZone:Surface object for this surface
    idf.newidfobject(
        "AirflowNetwork:MultiZone:Surface".upper(),
        Surface_Name=surface_name,
        Leakage_Component_Name=crack_name,
        External_Node_Name="Outdoors" if boundary_condition.lower() == "outdoors" else "",
    )


In [5]:
idf.save("test.idf")

In [5]:
# Define reusable crack names for different surface types
crack_templates = {
    "external": "ExternalWallCrack",
    "internal": "InternalWallCrack",
    "floor_ceiling": "FloorCeilingCrack",
    "roof": "RoofCrack",
}

# Classify surface types
def classify_surface(surface_type, boundary_condition):
    if surface_type.lower() == "wall":
        return "external" if boundary_condition.lower() == "outdoors" else "internal"
    elif surface_type.lower() in ["floor", "ceiling"]:
        return "floor_ceiling"
    elif surface_type.lower() == "roof":
        return "roof"
    else:
        return None

# Loop through all BuildingSurface:Detailed objects
for surface in idf.idfobjects["BUILDINGSURFACE:DETAILED"]:
    surface_name = surface.Name
    zone_name = surface.Zone_Name
    surface_type = surface.Surface_Type
    boundary_condition = surface.Outside_Boundary_Condition

    # Classify surface and get the crack template
    surface_class = classify_surface(surface_type, boundary_condition)
    if not surface_class:
        continue

    crack_name = crack_templates[surface_class]

    # Add AirflowNetwork:MultiZone:Surface object
    idf.newidfobject(
        "AirflowNetwork:MultiZone:Surface".upper(),
        Surface_Name=surface_name,
        Leakage_Component_Name=crack_name,
        External_Node_Name="Outdoors" if boundary_condition.lower() == "outdoors" else "",
    )


In [19]:
idf.newidfobject(
    "AirflowNetwork:MultiZone:Surface".upper(),
    Surface_Name="Surface_1",
    Leakage_Component_Name="CR-1",
    External_Node_Name = "SFacade",
)



BadEPFieldError: unknown field Name

In [22]:
processed_surfaces = set()

for surface in idf.idfobjects["BUILDINGSURFACE:DETAILED"]:
    surface_name = surface.Name
    zone_name = surface.Zone_Name
    boundary_condition = surface.Outside_Boundary_Condition
    boundary_object = surface.Outside_Boundary_Condition_Object

    # Only process internal surfaces (Zone-to-Zone)
    if boundary_condition.lower() == "surface" and boundary_object:
        # Create a unique key for this surface pair
        surface_pair = tuple(sorted([surface_name, boundary_object]))

        if surface_pair not in processed_surfaces:
            # Add AirflowNetwork definition for this surface
            idf.newidfobject(
                "AIRFLOWNETWORK:MULTIZONE:SURFACE",
                Surface_Name=f"{surface_name}_Airflow",
                Outside_Boundary_Condition_Object=boundary_object,
                AirflowNetwork_Multizone_Component_Name="InternalWallCrack",
            )

            # Mark this surface pair as processed
            processed_surfaces.add(surface_pair)


BadEPFieldError: unknown field Outside_Boundary_Condition_Object